In [ ]:
# ml_training.py - обучение моделей
import pandas as pd
import numpy as np
import sqlite3
from pathlib import Path
import json
import joblib
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Установка: pip install catboost lightgbm scikit-learn mlflow evidently sentence-transformers networkx plotly kaleido

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, f1_score
from sklearn.preprocessing import LabelEncoder
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sentence_transformers import SentenceTransformer
import mlflow
import mlflow.sklearn

# ======================== КОНФИГУРАЦИЯ ========================
DB_PATH = "educational_materials.db"
MODELS_DIR = Path("models")
MODELS_DIR.mkdir(exist_ok=True)

EMBEDDING_MODEL = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

# ======================== ЗАГРУЗКА ДАННЫХ ========================
def load_training_data():
    """Загрузка данных с метками из кластеризации"""
    
    conn = sqlite3.connect(DB_PATH)
    
    df = pd.read_sql_query("""
        SELECT 
            id,
            subject,
            topic,
            annotation,
            text_content,
            LENGTH(text_content) as text_length,
            LENGTH(text_content) - LENGTH(REPLACE(text_content, ' ', '')) + 1 as word_count,
            is_approved,
            is_generated,
            prev_material_id,
            next_material_id,
            parallel_cluster,
            sequential_cluster,
            complexity_cluster
        FROM materials
        WHERE parallel_cluster IS NOT NULL
    """, conn)
    
    conn.close()
    
    # Генерация эмбеддингов
    print("🔄 Генерация эмбеддингов...")
    texts = df['topic'].fillna('') + ' ' + df['annotation'].fillna('')
    embeddings = EMBEDDING_MODEL.encode(texts.tolist(), show_progress_bar=True)
    
    # Добавляем эмбеддинги как фичи
    for i in range(embeddings.shape[1]):
        df[f'emb_{i}'] = embeddings[:, i]
    
    return df

def engineer_features(df):
    """Инжиниринг признаков"""
    
    # 1. Текстовые признаки
    df['avg_word_length'] = df['text_length'] / df['word_count']
    df['has_prev'] = (df['prev_material_id'].notna()).astype(int)
    df['has_next'] = (df['next_material_id'].notna()).astype(int)
    
    # 2. Предметные признаки (one-hot encoding)
    subject_dummies = pd.get_dummies(df['subject'], prefix='subject')
    df = pd.concat([df, subject_dummies], axis=1)
    
    # 3. Плотность терминов (капитализированные слова)
    df['term_density'] = df['text_content'].str.count(r'[A-ZА-Я]{3,}') / df['word_count']
    df['term_density'] = df['term_density'].fillna(0)
    
    # 4. Читаемость (простая эвристика - средняя длина предложения)
    df['sentence_count'] = df['text_content'].str.count(r'[.!?]') + 1
    df['avg_sentence_length'] = df['word_count'] / df['sentence_count']
    df['avg_sentence_length'] = df['avg_sentence_length'].fillna(0)
    
    return df

# ======================== ОБУЧЕНИЕ МОДЕЛЕЙ ========================
class MultiTaskClassifier:
    """Обертка для обучения трех моделей"""
    
    def __init__(self):
        self.models = {
            'parallel': None,
            'sequential': None,
            'complexity': None
        }
        self.feature_names = None
        self.label_encoders = {}
    
    def prepare_features(self, df, target_col):
        """Подготовка фичей для обучения"""
        
        # Исключаем целевые переменные и ID
        exclude_cols = ['id', 'subject', 'topic', 'annotation', 'text_content', 
                       'parallel_cluster', 'sequential_cluster', 'complexity_cluster',
                       'prev_material_id', 'next_material_id', 'created_at', 'updated_at']
        
        feature_cols = [col for col in df.columns if col not in exclude_cols]
        
        X = df[feature_cols].fillna(0)
        y = df[target_col]
        
        # Label encoding для целевой переменной
        if target_col not in self.label_encoders:
            le = LabelEncoder()
            y = le.fit_transform(y)
            self.label_encoders[target_col] = le
        else:
            y = self.label_encoders[target_col].transform(y)
        
        return X, y, feature_cols
    
    def train_single_task(self, X_train, y_train, X_test, y_test, task_name):
        """Обучение одной задачи с перебором алгоритмов"""
        
        print(f"\n{'='*60}")
        print(f"🎯 Обучение модели: {task_name}")
        print(f"{'='*60}\n")
        
        # Три алгоритма для сравнения
        models = {
            'RandomForest': RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42),
            'LightGBM': LGBMClassifier(n_estimators=100, max_depth=7, learning_rate=0.1, random_state=42, verbose=-1),
            'CatBoost': CatBoostClassifier(iterations=100, depth=6, learning_rate=0.1, random_seed=42, verbose=False)
        }
        
        results = {}
        
        mlflow.set_experiment(f"educational_materials_{task_name}")
        
        for model_name, model in models.items():
            print(f"📊 Тестирование: {model_name}")
            
            with mlflow.start_run(run_name=f"{model_name}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"):
                
                # Обучение
                model.fit(X_train, y_train)
                
                # Предсказания
                y_pred = model.predict(X_test)
                y_pred_proba = model.predict_proba(X_test)
                
                # Метрики
                report = classification_report(y_test, y_pred, output_dict=True)
                
                accuracy = report['accuracy']
                weighted_f1 = report['weighted avg']['f1-score']
                weighted_precision = report['weighted avg']['precision']
                weighted_recall = report['weighted avg']['recall']
                
                # ROC-AUC (multiclass)
                if len(np.unique(y_test)) > 2:
                    roc_auc = roc_auc_score(y_test, y_pred_proba, multi_class='ovr', average='weighted')
                else:
                    roc_auc = roc_auc_score(y_test, y_pred_proba[:, 1])
                
                # Cross-validation
                cv_scores = cross_val_score(model, X_train, y_train, cv=3, scoring='f1_weighted')
                
                results[model_name] = {
                    'model': model,
                    'accuracy': accuracy,
                    'precision': weighted_precision,
                    'recall': weighted_recall,
                    'f1_score': weighted_f1,
                    'roc_auc': roc_auc,
                    'cv_mean': cv_scores.mean(),
                    'cv_std': cv_scores.std()
                }
                
                # Логирование в MLflow
                mlflow.log_param("model_type", model_name)
                mlflow.log_metric("accuracy", accuracy)
                mlflow.log_metric("precision", weighted_precision)
                mlflow.log_metric("recall", weighted_recall)
                mlflow.log_metric("f1_score", weighted_f1)
                mlflow.log_metric("roc_auc", roc_auc)
                mlflow.log_metric("cv_mean", cv_scores.mean())
                mlflow.log_metric("cv_std", cv_scores.std())
                
                mlflow.sklearn.log_model(model, f"{task_name}_{model_name}")
                
                print(f"  ✅ Accuracy: {accuracy:.4f}")
                print(f"  ✅ F1-Score: {weighted_f1:.4f}")
                print(f"  ✅ ROC-AUC: {roc_auc:.4f}")
                print(f"  ✅ CV: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}\n")
        
        # Выбор лучшей модели
        best_model_name = max(results, key=lambda x: results[x]['f1_score'])
        best_model = results[best_model_name]
        
        print(f"🏆 Лучшая модель: {best_model_name}")
        print(f"   F1-Score: {best_model['f1_score']:.4f}")
        print(f"   ROC-AUC: {best_model['roc_auc']:.4f}")
        
        # Сохранение
        model_path = MODELS_DIR / f"{task_name}_model.pkl"
        joblib.dump(best_model['model'], model_path)
        
        # Сохранение метаданных
        metadata = {
            'task': task_name,
            'best_model': best_model_name,
            'all_results': {k: {key: val for key, val in v.items() if key != 'model'} 
                           for k, v in results.items()},
            'trained_at': datetime.now().isoformat(),
            'model_path': str(model_path)
        }
        
        with open(MODELS_DIR / f"{task_name}_metadata.json", 'w') as f:
            json.dump(metadata, f, indent=2)
        
        return best_model['model'], results
    
    def train_all(self, df):
        """Обучение всех трех моделей"""
        
        df = engineer_features(df)
        
        tasks = [
            ('parallel', 'parallel_cluster'),
            ('sequential', 'sequential_cluster'),
            ('complexity', 'complexity_cluster')
        ]
        
        all_results = {}
        
        for task_name, target_col in tasks:
            
            X, y, feature_cols = self.prepare_features(df, target_col)
            
            if self.feature_names is None:
                self.feature_names = feature_cols
            
            # Split
            X_train, X_test, y_train, y_test = train_test_split(
                X, y, test_size=0.2, random_state=42, stratify=y
            )
            
            # Обучение
            model, results = self.train_single_task(
                X_train, y_train, X_test, y_test, task_name
            )
            
            self.models[task_name] = model
            all_results[task_name] = results
        
        # Сохранение label encoders
        joblib.dump(self.label_encoders, MODELS_DIR / 'label_encoders.pkl')
        joblib.dump(self.feature_names, MODELS_DIR / 'feature_names.pkl')
        
        print("\n✅ Все модели обучены и сохранены!")
        return all_results

# ======================== НЕПРЕРЫВНОЕ ОБУЧЕНИЕ ========================
class ContinuousLearningAgent:
    """Агент непрерывного обучения"""
    
    def __init__(self):
        self.models_dir = MODELS_DIR
        self.version_log = []
        self.drift_threshold = 0.15  # Порог для дрейфа
    
    def detect_data_drift(self, reference_data, current_data):
        """Обнаружение дрейфа данных через Evidently"""
        
        from evidently.report import Report
        from evidently.metric_preset import DataDriftPreset, DataQualityPreset
        
        print("🔍 Проверка дрейфа данных...")
        
        # Создание отчета
        report = Report(metrics=[
            DataDriftPreset(),
        ])
        
        report.run(reference_data=reference_data, current_data=current_data)
        
        # Сохранение отчета
        report.save_html(str(MODELS_DIR / f"drift_report_{datetime.now().strftime('%Y%m%d_%H%M%S')}.html"))
        
        # Извлечение метрик дрейфа
        report_dict = report.as_dict()
        
        # Упрощенная проверка: доля дрейфующих признаков
        drift_metrics = report_dict['metrics'][0]['result']
        drift_share = drift_metrics.get('share_of_drifted_columns', 0)
        
        print(f"   Доля дрейфующих признаков: {drift_share:.2%}")
        
        return drift_share > self.drift_threshold, drift_share
    
    def incremental_training(self, model, X_new, y_new, task_name):
        """Дообучение модели на новых данных"""
        
        print(f"🔄 Дообучение модели: {task_name}")
        
        # Для моделей, поддерживающих инкрементальное обучение
        if hasattr(model, 'partial_fit'):
            model.partial_fit(X_new, y_new)
        elif isinstance(model, (CatBoostClassifier, LGBMClassifier)):
            # Для бустинговых моделей - продолжаем обучение
            model.fit(X_new, y_new, init_model=model)
        else:
            # Для остальных - переобучение на объединенных данных
            print("   ⚠️ Модель не поддерживает инкрементальное обучение, выполняется полное переобучение")
            return None
        
        return model
    
    def full_retraining(self, df):
        """Полное переобучение всех моделей"""
        
        print("🔄 Полное переобучение моделей...")
        
        classifier = MultiTaskClassifier()
        results = classifier.train_all(df)
        
        # Логирование
        self.log_version({
            'type': 'full_retraining',
            'timestamp': datetime.now().isoformat(),
            'reason': 'Data drift detected',
            'results': results
        })
        
        return classifier
    
    def auto_update_pipeline(self, new_data_threshold=50):
        """Автоматический пайплайн обновления"""
        
        print("\n" + "="*60)
        print("🤖 АГЕНТ НЕПРЕРЫВНОГО ОБУЧЕНИЯ")
        print("="*60 + "\n")
        
        # 1. Загрузка текущих данных
        df_current = load_training_data()
        
        # 2. Проверка новых данных
        conn = sqlite3.connect(DB_PATH)
        new_count = pd.read_sql_query("""
            SELECT COUNT(*) as count 
            FROM materials 
            WHERE datetime(updated_at) > datetime('now', '-1 day')
        """, conn).iloc[0]['count']
        conn.close()
        
        print(f"📊 Новых материалов за последние 24ч: {new_count}")
        
        if new_count < new_data_threshold:
            print("✅ Недостаточно новых данных для обновления")
            return
        
        # 3. Загрузка референсных данных (для проверки дрейфа)
        try:
            df_reference = pd.read_csv(MODELS_DIR / 'reference_data.csv')
        except FileNotFoundError:
            print("📝 Создание референсного датасета...")
            df_reference = df_current.copy()
            df_reference.to_csv(MODELS_DIR / 'reference_data.csv', index=False)
        
        # 4. Проверка дрейфа
        df_current_eng = engineer_features(df_current)
        df_reference_eng = engineer_features(df_reference)
        
        # Берем только числовые столбцы для сравнения
        numeric_cols = df_current_eng.select_dtypes(include=[np.number]).columns.tolist()
        
        has_drift, drift_score = self.detect_data_drift(
            df_reference_eng[numeric_cols],
            df_current_eng[numeric_cols]
        )
        
        # 5. Решение о переобучении
        if has_drift:
            print(f"⚠️ ДРЕЙФ ОБНАРУЖЕН (score: {drift_score:.2%}) - выполняется полное переобучение")
            self.full_retraining(df_current)
            
            # Обновление референсного датасета
            df_current.to_csv(MODELS_DIR / 'reference_data.csv', index=False)
        else:
            print("✅ Дрейфа нет - выполняется дообучение")
            
            # Загрузка текущих моделей
            for task in ['parallel', 'sequential', 'complexity']:
                model = joblib.load(MODELS_DIR / f"{task}_model.pkl")
                
                # Подготовка новых данных
                X_new, y_new, _ = MultiTaskClassifier().prepare_features(
                    df_current_eng, f"{task}_cluster"
                )
                
                # Дообучение
                updated_model = self.incremental_training(
                    model, X_new, y_new, task
                )
                
                if updated_model:
                    joblib.dump(updated_model, MODELS_DIR / f"{task}_model.pkl")
            
            self.log_version({
                'type': 'incremental_update',
                'timestamp': datetime.now().isoformat(),
                'new_samples': new_count,
                'drift_score': drift_score
            })
        
        print("\n✅ Обновление завершено!")
    
    def log_version(self, info):
        """Логирование версий моделей"""
        
        log_file = MODELS_DIR / 'version_log.json'
        
        if log_file.exists():
            with open(log_file, 'r') as f:
                log = json.load(f)
        else:
            log = []
        
        log.append(info)
        
        with open(log_file, 'w') as f:
            json.dump(log, f, indent=2)
        
        print(f"📝 Версия залогирована: {log_file}")

# ======================== ОЦЕНКА ВРЕМЕНИ ОСВОЕНИЯ ========================
class LearningTimeEstimator:
    """Агент оценки времени освоения"""
    
    def __init__(self):
        # Базовые временные параметры (часы)
        self.base_time = {
            'simple': 2,    # Простой материал
            'medium': 4,    # Средний
            'complex': 8    # Сложный
        }
        
        # Коэффициенты
        self.parallel_coef = 0.7  # Параллельное изучение экономит 30% времени
        self.sequential_penalty = 1.2  # Последовательное добавляет 20% (переключение контекста)
    
    def load_predictions(self):
        """Загрузка предсказаний моделей"""
        
        df = load_training_data()
        df = engineer_features(df)
        
        # Загрузка моделей
        models = {}
        for task in ['parallel', 'sequential', 'complexity']:
            models[task] = joblib.load(MODELS_DIR / f"{task}_model.pkl")
        
        label_encoders = joblib.load(MODELS_DIR / 'label_encoders.pkl')
        feature_names = joblib.load(MODELS_DIR / 'feature_names.pkl')
        
        X = df[feature_names].fillna(0)
        
        # Предсказания
        for task in ['parallel', 'sequential', 'complexity']:
            pred = models[task].predict(X)
            pred_labels = label_encoders[f"{task}_cluster"].inverse_transform(pred)
            df[f'{task}_pred'] = pred_labels
        
        return df
    
    def estimate_material_time(self, row):
        """Оценка времени для одного материала"""
        
        complexity_map = {0: 'simple', 1: 'medium', 2: 'complex'}
        complexity = complexity_map.get(row['complexity_pred'], 'medium')
        
        base = self.base_time[complexity]
        
        # Корректировка по длине текста
        length_factor = min(row['word_count'] / 1000, 3)  # макс 3x
        
        time_hours = base * length_factor
        
        return time_hours
    
    def estimate_subject_time(self, df, subject):
        """Оценка времени для предмета"""
        
        subject_materials = df[df['subject'] == subject]
        
        total_time = 0
        parallel_groups = subject_materials.groupby('parallel_pred')
        
        for group_id, group in parallel_groups:
            # Материалы в группе изучаются параллельно
            max_time_in_group = max([self.estimate_material_time(row) 
                                     for _, row in group.iterrows()])
            total_time += max_time_in_group * self.parallel_coef
        
        return total_time
    
    def estimate_curriculum_time(self, df, subjects_list, parallel_subjects=None):
        """Оценка времени для набора предметов"""
        
        if parallel_subjects is None:
            parallel_subjects = []
        
        total_time = 0
        
        # Последовательные предметы
        sequential = [s for s in subjects_list if s not in parallel_subjects]
        for subject in sequential:
            time = self.estimate_subject_time(df, subject)
            total_time += time * self.sequential_penalty
        
        # Параллельные предметы (берем максимум)
        if parallel_subjects:
            parallel_times = [self.estimate_subject_time(df, s) for s in parallel_subjects]
            total_time += max(parallel_times) * self.parallel_coef
        
        return total_time
    
    def visualize_time_estimates(self, df):
        """Визуализация временных оценок"""
        
        import plotly.express as px
        import plotly.graph_objects as go
        from plotly.subplots import make_subplots
        
        # 1. Время по предметам
        subject_times = {}
        for subject in df['subject'].unique():
            subject_times[subject] = self.estimate_subject_time(df, subject)
        
        fig1 = px.bar(
            x=list(subject_times.keys()),
            y=list(subject_times.values()),
            title="Оценка времени освоения предметов (часы)",
            labels={'x': 'Предмет', 'y': 'Часы'},
            color=list(subject_times.values()),
            color_continuous_scale='Blues'
        )
        fig1.write_html(MODELS_DIR / 'time_by_subject.html')
        
        # 2. Распределение по сложности
        df['time_estimate'] = df.apply(self.estimate_material_time, axis=1)
        
        fig2 = px.box(
            df,
            x='complexity_pred',
            y='time_estimate',
            title="Распределение времени по уровню сложности",
            labels={'complexity_pred': 'Сложность', 'time_estimate': 'Часы'}
        )
        fig2.write_html(MODELS_DIR / 'time_by_complexity.html')
        
        # 3. Временная шкала предметов
        cumulative_time = []
        cumulative = 0
        for subject, time in subject_times.items():
            cumulative += time
            cumulative_time.append({'subject': subject, 'cumulative_hours': cumulative})
        
        cumulative_df = pd.DataFrame(cumulative_time)
        
        fig3 = px.line(
            cumulative_df,
            x='subject',
            y='cumulative_hours',
            title="Кумулятивное время освоения",
            markers=True
        )
        fig3.write_html(MODELS_DIR / 'cumulative_time.html')
        
        print("📊 Визуализации сохранены:")
        print(f"   - {MODELS_DIR / 'time_by_subject.html'}")
        print(f"   - {MODELS_DIR / 'time_by_complexity.html'}")
        print(f"   - {MODELS_DIR / 'cumulative_time.html'}")
        
        return subject_times

# ======================== РЕКОМЕНДАТЕЛЬНАЯ СИСТЕМА ========================
class LearningPathAgent:
    """Агент построения индивидуальных траекторий"""
    
    def __init__(self):
        self.time_estimator = LearningTimeEstimator()
    
    def collect_user_info(self):
        """Сбор информации от пользователя"""
        
        print("\n" + "="*60)
        print("🎓 ПОСТРОЕНИЕ ИНДИВИДУАЛЬНОЙ ТРАЕКТОРИИ ОБУЧЕНИЯ")
        print("="*60 + "\n")
        
        # 1. Изученные предметы
        print("Какие предметы вы уже изучили? (через запятую, Enter если нет)")
        completed = input("> ").strip()
        completed_subjects = [s.strip() for s in completed.split(',')] if completed else []
        
        # 2. Временные рамки
        print("\nСколько недель у вас на освоение курса?")
        total_weeks = int(input("> "))
        
        print("\nСколько часов в день вы можете уделять изучению?")
        hours_per_day = float(input("> "))
        
        # 3. Предпочтения
        print("\nПриоритет: быстрое освоение (1) или глубокое изучение (2)?")
        priority = int(input("> "))
        
        return {
            'completed_subjects': completed_subjects,
            'total_weeks': total_weeks,
            'hours_per_day': hours_per_day,
            'total_hours': total_weeks * 7 * hours_per_day,
            'priority': 'fast' if priority == 1 else 'deep'
        }
    
    def build_learning_path(self, df, user_info):
        """Построение траектории обучения"""
        
        available_subjects = [s for s in df['subject'].unique() 
                             if s not in user_info['completed_subjects']]
        
        total_budget = user_info['total_hours']
        
        # Оценка времени для каждого предмета
        subject_times = {}
        for subject in available_subjects:
            subject_times[subject] = self.time_estimator.estimate_subject_time(df, subject)
        
        # Анализ зависимостей между предметами
        dependencies = self._analyze_dependencies(df)
        
        # Построение траектории с учетом бюджета
        path = self._optimize_path(
            subject_times, 
            dependencies, 
            total_budget,
            user_info['priority']
        )
        
        return path, subject_times
    
    def _analyze_dependencies(self, df):
        """Анализ зависимостей между предметами"""
        
        # Простая эвристика: предметы связаны, если есть материалы с prev/next из разных предметов
        dependencies = {}
        
        for subject in df['subject'].unique():
            dependencies[subject] = {'prerequisites': [], 'enables': []}
        
        # Анализ связей
        for _, row in df.iterrows():
            if pd.notna(row['prev_material_id']):
                prev_row = df[df['id'] == row['prev_material_id']]
                if not prev_row.empty and prev_row.iloc[0]['subject'] != row['subject']:
                    prereq = prev_row.iloc[0]['subject']
                    if prereq not in dependencies[row['subject']]['prerequisites']:
                        dependencies[row['subject']]['prerequisites'].append(prereq)
        
        return dependencies
    
    def _optimize_path(self, subject_times, dependencies, budget, priority):
        """Оптимизация траектории с учетом ограничений"""
        
        import networkx as nx
        
        # Строим граф зависимостей
        G = nx.DiGraph()
        
        for subject, time in subject_times.items():
            G.add_node(subject, time=time)
        
        for subject, deps in dependencies.items():
            for prereq in deps['prerequisites']:
                G.add_edge(prereq, subject)
        
        # Топологическая сортировка (учитываем зависимости)
        try:
            topo_order = list(nx.topological_sort(G))
        except nx.NetworkXError:
            # Если есть циклы - используем простой порядок
            topo_order = list(subject_times.keys())
        
        # Жадный алгоритм: выбираем предметы по приоритету
        if priority == 'fast':
            # Сортируем по времени (быстрые первые)
            topo_order = sorted(topo_order, key=lambda x: subject_times[x])
        else:
            # Оставляем топологический порядок
            pass
        
        # Формируем путь с учетом бюджета
        path = {
            'weeks': []
        }
        
        remaining_budget = budget
        current_week = []
        week_hours = 0
        week_number = 1
        
        for subject in topo_order:
            subject_time = subject_times[subject]
            
            if subject_time > remaining_budget:
                break
            
            # Проверяем, можно ли добавить в текущую неделю
            if week_hours + subject_time <= budget / 4:  # ~25% бюджета на неделю
                current_week.append({
                    'subject': subject,
                    'hours': subject_time,
                    'parallel': len(current_week) > 0
                })
                week_hours += subject_time
            else:
                # Начинаем новую неделю
                if current_week:
                    path['weeks'].append({
                        'week_number': week_number,
                        'subjects': current_week,
                        'total_hours': week_hours
                    })
                    week_number += 1
                
                current_week = [{
                    'subject': subject,
                    'hours': subject_time,
                    'parallel': False
                }]
                week_hours = subject_time
            
            remaining_budget -= subject_time
        
        # Добавляем последнюю неделю
        if current_week:
            path['weeks'].append({
                'week_number': week_number,
                'subjects': current_week,
                'total_hours': week_hours
            })
        
        return path
    
    def visualize_learning_path(self, path, subject_times):
        """Визуализация траектории обучения"""
        
        import plotly.figure_factory as ff
        import plotly.graph_objects as go
        
        # 1. Gantt chart
        gantt_data = []
        
        start_day = 0
        for week_info in path['weeks']:
            for subject_info in week_info['subjects']:
                gantt_data.append({
                    'Task': subject_info['subject'],
                    'Start': f"Day {start_day}",
                    'Finish': f"Day {start_day + 7}",
                    'Resource': 'Week ' + str(week_info['week_number'])
                })
            start_day += 7
        
        if gantt_data:
            # Преобразуем в даты
            from datetime import datetime, timedelta
            base_date = datetime(2024, 1, 1)
            
            for item in gantt_data:
                start_days = int(item['Start'].split()[1])
                finish_days = int(item['Finish'].split()[1])
                item['Start'] = (base_date + timedelta(days=start_days)).strftime('%Y-%m-%d')
                item['Finish'] = (base_date + timedelta(days=finish_days)).strftime('%Y-%m-%d')
            
            fig1 = ff.create_gantt(
                gantt_data,
                index_col='Resource',
                show_colorbar=True,
                group_tasks=True,
                title="Траектория обучения (временная шкала)"
            )
            fig1.write_html(MODELS_DIR / 'learning_path_gantt.html')
        
        # 2. Граф зависимостей
        import networkx as nx
        
        G = nx.DiGraph()
        
        for week_info in path['weeks']:
            for i, subj1 in enumerate(week_info['subjects']):
                G.add_node(subj1['subject'], week=week_info['week_number'])
                
                # Связи внутри недели (параллельные)
                for subj2 in week_info['subjects'][i+1:]:
                    G.add_edge(subj1['subject'], subj2['subject'], type='parallel')
        
        # Связи между неделями (последовательные)
        for i in range(len(path['weeks']) - 1):
            for subj1 in path['weeks'][i]['subjects']:
                for subj2 in path['weeks'][i+1]['subjects']:
                    G.add_edge(subj1['subject'], subj2['subject'], type='sequential')
        
        # Визуализация через Plotly
        pos = nx.spring_layout(G, seed=42)
        
        edge_trace = []
        for edge in G.edges():
            x0, y0 = pos[edge[0]]
            x1, y1 = pos[edge[1]]
            edge_type = G.edges[edge]['type']
            
            edge_trace.append(
                go.Scatter(
                    x=[x0, x1, None],
                    y=[y0, y1, None],
                    mode='lines',
                    line=dict(width=2, color='blue' if edge_type == 'sequential' else 'green'),
                    hoverinfo='none',
                    showlegend=False
                )
            )
        
        node_trace = go.Scatter(
            x=[pos[node][0] for node in G.nodes()],
            y=[pos[node][1] for node in G.nodes()],
            mode='markers+text',
            text=[node for node in G.nodes()],
            textposition='top center',
            marker=dict(
                size=20,
                color=[G.nodes[node]['week'] for node in G.nodes()],
                colorscale='Viridis',
                showscale=True,
                colorbar=dict(title="Неделя")
            ),
            hoverinfo='text'
        )
        
        fig2 = go.Figure(data=edge_trace + [node_trace])
        fig2.update_layout(
            title="Граф траектории обучения",
            showlegend=False,
            hovermode='closest',
            xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
            yaxis=dict(showgrid=False, zeroline=False, showticklabels=False)
        )
        fig2.write_html(MODELS_DIR / 'learning_path_graph.html')
        
        print("\n📊 Визуализации траектории сохранены:")
        print(f"   - {MODELS_DIR / 'learning_path_gantt.html'}")
        print(f"   - {MODELS_DIR / 'learning_path_graph.html'}")
        
        # Вывод текстового плана
        print("\n📅 ПЛАН ОБУЧЕНИЯ:\n")
        total_hours = 0
        for week_info in path['weeks']:
            print(f"Неделя {week_info['week_number']} ({week_info['total_hours']:.1f} часов):")
            for subj in week_info['subjects']:
                parallel_mark = " [параллельно]" if subj['parallel'] else ""
                print(f"  - {subj['subject']}: {subj['hours']:.1f}ч{parallel_mark}")
            total_hours += week_info['total_hours']
        
        print(f"\n✅ Общее время: {total_hours:.1f} часов")

# ======================== ГЕНЕРАЦИЯ ОТЧЕТА ========================
def generate_module_c_report(all_results, path_info):
    """Генерация итогового отчета по модулю В"""
    
    report_md = f"""
# ОТЧЕТ ПО МОДУЛЮ В: ML-МОДЕЛИ И РЕКОМЕНДАТЕЛЬНАЯ СИСТЕМА

Дата формирования: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

---

## 1. ОБУЧЕННЫЕ МОДЕЛИ

### 1.1 Параллельное изучение

**Задача:** Классификация материалов, подходящих для совместного изучения

**Протестированные алгоритмы:**
"""
    
    for task in ['parallel', 'sequential', 'complexity']:
        task_results = all_results[task]
        best_model = max(task_results, key=lambda x: task_results[x]['f1_score'])
        
        report_md += f"\n#### {task.capitalize()}\n\n"
        report_md += "| Алгоритм | Accuracy | Precision | Recall | F1-Score | ROC-AUC | CV Mean |\n"
        report_md += "|----------|----------|-----------|--------|----------|---------|----------|\n"
        
        for model_name, metrics in task_results.items():
            report_md += f"| {model_name} | {metrics['accuracy']:.4f} | {metrics['precision']:.4f} | {metrics['recall']:.4f} | {metrics['f1_score']:.4f} | {metrics['roc_auc']:.4f} | {metrics['cv_mean']:.4f} |\n"
        
        report_md += f"\n**Выбранная модель:** {best_model}\n\n"
        report_md += f"**Обоснование:** {best_model} показал лучшие результаты по F1-Score ({task_results[best_model]['f1_score']:.4f}) и ROC-AUC ({task_results[best_model]['roc_auc']:.4f}) с стабильной кросс-валидацией.\n\n"
    
    report_md += """
---

## 2. НЕПРЕРЫВНОЕ ОБУЧЕНИЕ

### 2.1 Архитектура системы